# Bank Term Deposit Classification

## Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV
from sklearn.tree import plot_tree
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Metrics
from sklearn.metrics import balanced_accuracy_score, precision_score, recall_score, make_scorer

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from tabpfn import TabPFNClassifier

# Class imbalance
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler, SMOTE


c:\Users\Alumne_mati1\Documents\machine_learning_course_cifo_2026\.venv_py312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Data

In [2]:
# Load the dataset
df = pd.read_csv('../data/bank_term_deposit/bank_term_deposit_prepared.csv')
df.head()

,id,split,age,job,marital,education,default,balance,housing,loan,...,day_sin,day_cos,early_month,late_month,avg_contact_duration,contacts_per_month,financial_stress,education_unknown,age_group,balance_group
0,1,labeled,31,management,married,tertiary,0,460,0,0,...,-0.571268,0.820763,0,1,0.928571,1.750,0,0,1,2
1,2,labeled,34,blue-collar,married,secondary,0,1826,1,0,...,-0.790776,-0.612106,0,1,101.500000,0.400,0,0,1,3
2,3,labeled,50,blue-collar,married,secondary,0,290,1,0,...,0.988468,0.151428,1,0,75.333333,0.375,0,0,3,2
3,4,labeled,42,admin.,divorced,secondary,0,1077,1,0,...,0.299363,-0.954139,0,0,213.000000,0.200,0,0,2,3
4,5,labeled,47,services,single,secondary,0,41,1,0,...,0.848644,0.528964,1,0,298.000000,0.200,0,0,3,2


In [3]:
df.shape

(3063, 32)

## Split Column

In [4]:
leaderboard_df = df[df['split'] == 'leaderboard'].copy()
df = df[df['split'] == 'labeled']

In [5]:
df['y'].value_counts()

y
0    1833
1     174
Name: count, dtype: int64

## Functions

In [6]:
def rs_cv_results(rs, n_best=5):
    metric_name = 'Balanced Accuracy'
    # Get the indices of the smallest Bacc values
    best_indices = np.argsort(rs.cv_results_[f'mean_test_{metric_name}'])[-n_best:]

    # Show best Baccs and their corresponding hyperparams.
    for i in best_indices[::-1]:
        print(f'{metric_name}:', rs.cv_results_[f'mean_test_{metric_name}'][i].round(2))
        print('Hyperparams:', rs.cv_results_['params'][i])
        print()

In [7]:
def create_submission(model, X, y, leaderboard_df, fts2drop, decision_th=None, fpath=''):
    X_lb = leaderboard_df.drop(columns=fts2drop)
    model.fit(X, y)
    if decision_th is None:
        preds = model.predict(X_lb)
    else:
        y_prob1 = model.predict_proba(X_lb)[:, 1]
        preds = (y_prob1 >= decision_th).astype(int)
    submission_df = pd.DataFrame({
        'id': leaderboard_df['id'],
        'prediction': preds
    })
    if not fpath:
        fpath = 'house_pricing_submission.csv'
    submission_df.to_csv(fpath, index=False)

## Preprocessing

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler, OrdinalEncoder, OneHotEncoder
from sklearn.base import BaseEstimator, TransformerMixin

In [9]:
preprocessor = ColumnTransformer(
    transformers=[
        # Ordinal: education has a natural order
        ('education', OrdinalEncoder(
            categories=[['unknown', 'primary', 'secondary', 'tertiary']]),
         ['education']),
        # One-hot: nominal categoricals (drop='first' to avoid collinearity)
        ('onehot', OneHotEncoder(
            drop='first', sparse_output=False, dtype=int,
            handle_unknown='ignore'),
         ['job', 'marital', 'contact', 'poutcome', 'season']),
    ],
    remainder='passthrough',  # keep all the numeric columns as-is
)

### Custom transformer for the constructed feature

Create `contact_intensity` feature. As it needs normalization, do it as a transformer so we can include it in a Pipeline and does not leak with validation or test data.

- `fit` only *learns* parameters that depend on the data &mdash; here the `MinMaxScaler` used by `contact_intensity`.
- `transform` *creates* the new columns, always working on a **copy** so the input is never mutated.

In [10]:
class ContactIntensityFeature(BaseEstimator, TransformerMixin):
    from sklearn.preprocessing import MinMaxScaler

    intensity_cols = ['campaign', 'previous', 'duration']

    def fit(self, X, y=None):
        # Learn the min/max of the contact columns from the training data only
        self.scaler_ = MinMaxScaler().fit(X[self.intensity_cols])
        return self

    def transform(self, X):
        X = X.copy()  # never mutate the caller's DataFrame

        # Contact intensity
        # Normalized so each feature contributes equally to the intensity score
        norm = self.scaler_.transform(X[self.intensity_cols])
        X['contact_intensity'] = norm.sum(axis=1)

        return X

## Train / Test

In [11]:
target_col = 'y'
fts2drop = ['id', 'split', target_col]

In [12]:
X = df.drop(columns=fts2drop)
y = df[target_col]

# Train / Test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    stratify=y,
    test_size=0.2,
    random_state=42
)

In [13]:
cat_cols = X.select_dtypes(include='object').columns
X_train[cat_cols] = X_train[cat_cols].astype('category')
X_test[cat_cols] = X_test[cat_cols].astype('category')

C:\Users\Alumne_mati1\AppData\Local\Temp\ipykernel_17552\987360480.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include='object').columns


## Cross-validation

In [14]:
# Declare KFold
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Declare scores to be used
scoring = {
    'Balanced Accuracy': make_scorer(balanced_accuracy_score),
    'Precision': make_scorer(precision_score, zero_division=0),
    'Recall': make_scorer(recall_score)
}

In [15]:
def print_metrics(cv_results):
    for sc in scoring.keys():
        print(f'Train {sc}:', cv_results[f'train_{sc}'].mean().round(3))
    print()
    for sc in scoring.keys():
        print(f'Validation {sc}:', cv_results[f'test_{sc}'].mean().round(3))
        
def print_metrics_rs(rand_search, idx=None):
    if idx is None:
        idx = rand_search.best_index_
    for sc in scoring.keys():
        print(f'Train {sc}:', rand_search.cv_results_[f'mean_train_{sc}'][idx].round(3))
    print()
    for sc in scoring.keys():
        print(f'Validation {sc}:', rand_search.cv_results_[f'mean_test_{sc}'][idx].round(3))

## Models

### Baseline 1

Doing something like this is not very informative...

In [16]:
from sklearn.dummy import DummyClassifier

In [17]:
bl1 = DummyClassifier(strategy='most_frequent')
bl1_cv = cross_validate(
    bl1,
    X_train,
    y_train,
    cv=kf,
    scoring=scoring,
    return_train_score=True
)

print_metrics(bl1_cv)

Train Balanced Accuracy: 0.5
Train Precision: 0.0
Train Recall: 0.0

Validation Balanced Accuracy: 0.5
Validation Precision: 0.0
Validation Recall: 0.0


In [18]:
bl1 = DummyClassifier(strategy='stratified')
bl1_cv = cross_validate(
    bl1,
    X_train,
    y_train,
    cv=kf,
    scoring=scoring,
    return_train_score=True
)

print_metrics(bl1_cv)

Train Balanced Accuracy: 0.497
Train Precision: 0.081
Train Recall: 0.082

Validation Balanced Accuracy: 0.496
Validation Precision: 0.066
Validation Recall: 0.073


### Baseline 2

In [19]:
bl2 = LogisticRegression(max_iter=10_000)
bl2_cv = cross_validate(
    bl2,
    X_train[['duration']],
    y_train,
    cv=kf,
    scoring=scoring,
    return_train_score=True
)

print_metrics(bl2_cv)

Train Balanced Accuracy: 0.712
Train Precision: 0.82
Train Recall: 0.432

Validation Balanced Accuracy: 0.712
Validation Precision: 0.821
Validation Recall: 0.433


### Logistic Regression

In [ ]:
lr_pipeline = Pipeline([
    ('contact_intensity', ContactIntensityFeature()),
    ('preprocessor', preprocessor),
    ('scaler', None),
    ('lr', LogisticRegression(
        solver='saga',
        max_iter=1_000,
    )),
])

params = [{
    'scaler': [MinMaxScaler(), RobustScaler(), StandardScaler()],
    'lr__C': np.logspace(-3, 2, 20),
    'lr__l1_ratio': np.arange(0, 1.01, 0.1),
    'lr__class_weight': [
        'balanced',
        None,
        {0:1, 1:2},
        {0:1, 1:3},
        {0:1, 1:5},
        {0:1, 1:10},
    ],
}]

lr_rs = RandomizedSearchCV(
    lr_pipeline,
    param_distributions=params,
    n_iter=50,
    scoring=scoring,
    cv=kf,
    refit='Balanced Accuracy',
    return_train_score=True,
    n_jobs=-1
)

lr_rs.fit(X_train, y_train)

print(lr_rs.best_params_)
print()
print_metrics_rs(lr_rs)

{'scaler': RobustScaler(), 'lr__l1_ratio': np.float64(0.9), 'lr__class_weight': {0: 1, 1: 2}, 'lr__C': np.float64(8.858667904100823)}

Train Balanced Accuracy: 0.758
Train Precision: 0.752
Train Recall: 0.533

Validation Balanced Accuracy: 0.761
Validation Precision: 0.757
Validation Recall: 0.54


c:\Users\Alumne_mati1\Documents\machine_learning_course_cifo_2026\.venv_py312\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [ ]:
joblib.dump(lr_rs.best_estimator_, '../models/bank/lr.joblib')

## Balance

### LR experiments

In [ ]:
lr_pipeline = joblib.load('../models/bank/lr.joblib')

With random undersampling it gets a bit better in recall in exchange of a lot of precision:

In [ ]:
lr_pipeline['lr'].get_params()

In [ ]:
lr_us = ImbPipeline([
    ('rus', RandomUnderSampler()),
    ('contact_intensity', ContactIntensityFeature()),
    ('preprocessor', preprocessor),
    ('scaler', lr_pipeline['scaler']),
    ('lr', LogisticRegression(**lr_pipeline['lr'].get_params())),
])

lr_us_cv = cross_validate(
    lr_us,
    X_train,
    y_train,
    cv=kf,
    scoring=scoring,
    return_train_score=True
)

print_metrics(lr_us_cv)

With random oversampling it gets a bit better in precision in exchange of some recall:

In [ ]:
lr_os = ImbPipeline ([
    ('ros', RandomOverSampler()),
    ('contact_intensity', ContactIntensityFeature()),
    ('preprocessor', preprocessor),
    ('scaler', lr_pipeline['scaler']),
    ('lr', LogisticRegression(**lr_pipeline['lr'].get_params())),
])

lr_os_cv = cross_validate(
    lr_os,
    X_train,
    y_train,
    cv=kf,
    scoring=scoring,
    return_train_score=True
)

print_metrics(lr_os_cv)

Threshold study with balanced accuracy for the LR model without any under or oversampling:

In [ ]:
y_prob1 = cross_val_predict(
    lr_pipeline,
    X_train,
    y_train,
    cv=kf,
    method='predict_proba'
)[:, 1]

ths = np.arange(0, 1.05, 0.05)
baccs = []

for th in ths:
    preds = (y_prob1 > th).astype(int)
    bacc = balanced_accuracy_score(y_train, preds)
    baccs.append(bacc)

In [ ]:
idx_best = np.argmax(baccs)
ths[idx_best], baccs[idx_best]

In [ ]:
plt.plot(ths, baccs, marker='o')
plt.xlabel('Threshold')
plt.ylabel('Balanced Accuracy')
plt.show()

### RF experiments

Some easy experiments just to see which methods seem to work well.

In [ ]:
# Standard, no class balance

rf_pipeline = Pipeline([
    ('contact_intensity', ContactIntensityFeature()),
    ('preprocessor', preprocessor),
    ('rf', RandomForestClassifier(max_depth=10, n_jobs=-1)),
])
     
rf_cv = cross_validate(
    rf_pipeline,
    X_train,
    y_train,
    cv=kf,
    scoring=scoring,
    return_train_score=True
)

print_metrics(rf_cv)

In [ ]:
# Class weight

rf_pipeline = Pipeline([
    ('contact_intensity', ContactIntensityFeature()),
    ('preprocessor', preprocessor),
    ('rf', RandomForestClassifier(max_depth=10, class_weight='balanced', n_jobs=-1)),
])

rf_cv = cross_validate(
    rf_pipeline,
    X_train,
    y_train,
    cv=kf,
    scoring=scoring,
    return_train_score=True
)

print_metrics(rf_cv)

In [ ]:
# Random Undersampling

rf_pipeline = ImbPipeline([
    ('rus', RandomUnderSampler()),
    ('contact_intensity', ContactIntensityFeature()),
    ('preprocessor', preprocessor),
    ('rf', RandomForestClassifier(max_depth=10, n_jobs=-1)),
])


rf_cv = cross_validate(
    rf_pipeline,
    X_train,
    y_train,
    cv=kf,
    scoring=scoring,
    return_train_score=True
)

print_metrics(rf_cv)

In [ ]:
# Random Oversampling

rf_pipeline = ImbPipeline([
    ('ros', RandomOverSampler()),
    ('contact_intensity', ContactIntensityFeature()),
    ('preprocessor', preprocessor),
    ('rf', RandomForestClassifier(max_depth=10, n_jobs=-1)),
])

rf_cv = cross_validate(
    rf_pipeline,
    X_train,
    y_train,
    cv=kf,
    scoring=scoring,
    return_train_score=True
)

print_metrics(rf_cv)

In [ ]:
# SMOTE

rf_pipeline = ImbPipeline([
    ('contact_intensity', ContactIntensityFeature()),
    ('preprocessor', preprocessor),
    ('smote', SMOTE()),
    ('rf', RandomForestClassifier(max_depth=10, n_jobs=-1)),
])

rf_cv = cross_validate(
    rf_pipeline,
    X_train,
    y_train,
    cv=kf,
    scoring=scoring,
    return_train_score=True
)

print_metrics(rf_cv)

### RF hyperparameter tuning

For maximizing balanced accuracy, it looked like undersampling was a good strategy.

Let's try hyperparameter search with some of the best options we tried.

In [ ]:
rf_pipeline = ImbPipeline([
    ('rus', RandomUnderSampler()),
    ('contact_intensity', ContactIntensityFeature()),
    ('preprocessor', preprocessor),
    ('rf', RandomForestClassifier(n_jobs=-1)),
])

params = [{
    'rf__n_estimators': [100, 200, 300, 400],
    'rf__max_depth': [2, 3, 5, 10, 20, 30, 50, None],
    'rf__class_weight': [
        'balanced',
        {0: 1, 1: 2},
        {0: 1, 1: 5},
        {0: 1, 1: 10},
        {0: 1, 1: 50},
    ]
}]

rf_rs = RandomizedSearchCV(
    rf_pipeline,
    n_iter=100,
    param_distributions=params,
    scoring=scoring,
    cv=kf,
    refit='Balanced Accuracy',
    return_train_score=True,
    n_jobs=-1
)

rf_rs.fit(X_train, y_train)

print(rf_rs.best_params_)
print()
print_metrics_rs(rf_rs)

In [ ]:
y_prob1 = cross_val_predict(
    rf_rs.best_estimator_,
    X_train,
    y_train,
    cv=kf,
    method='predict_proba'
)[:, 1]

ths = np.arange(0, 1.05, 0.05)
baccs = []

for th in ths:
    preds = (y_prob1 >= th).astype(int)
    bacc = balanced_accuracy_score(y_train, preds)
    baccs.append(bacc)

In [ ]:
idx_best = np.argmax(baccs)
ths[idx_best], baccs[idx_best]

In [ ]:
plt.plot(ths, baccs, marker='o')
plt.xlabel('Threshold')
plt.ylabel('Balanced Accuracy')
plt.show()

#### Feature importances

In [ ]:
rf_rs.best_estimator_.fit(X_train, y_train)

# Extract the feature names from the preprocessor
feature_names = rf_rs.best_estimator_.named_steps['preprocessor'].get_feature_names_out()

rf_ft_imps = pd.DataFrame({
    'feature': feature_names,
    'importance': rf_rs.best_estimator_['rf'].feature_importances_
}).sort_values('importance', ascending=False).round(3)

rf_ft_imps.head(20)

#### Save model

In [ ]:
joblib.dump(rf_rs.best_estimator_, '../models/bank/rf_ru.joblib')

### GB experiments

In [ ]:
# Without undersampling

gb_pipeline = Pipeline([
    ('contact_intensity', ContactIntensityFeature()),
    ('gb', HistGradientBoostingClassifier(loss='log_loss', random_state=42)),
])

params = {
    'gb__learning_rate': np.logspace(-3, 0, 10),
    'gb__min_samples_leaf': [1, 2, 4, 6],
    'gb__max_depth': [3, 4, 5, 6, 8, 10, 20],
    'gb__max_features': np.arange(0.1, 1.1, 0.1),
    'gb__max_leaf_nodes': [None, 5, 10, 20],
    'gb__validation_fraction': [0.1, 0.15, 0.2],
    'gb__class_weight': [
        'balanced',
        {0: 1, 1: 2},
        {0: 1, 1: 5},
        {0: 1, 1: 10},
        {0: 1, 1: 50},
    ]
}

gb_rs = RandomizedSearchCV(
    gb_pipeline,
    n_iter=200,
    param_distributions=params,
    scoring=scoring,
    cv=kf,
    refit='Balanced Accuracy',
    return_train_score=True,
    n_jobs=-1
)

gb_rs.fit(X_train, y_train)

print(gb_rs.best_params_)
print()
print_metrics_rs(gb_rs)

In [ ]:
# With understampling

gb_us_pipeline = ImbPipeline([
    ('rus', RandomUnderSampler()),
    ('contact_intensity', ContactIntensityFeature()),
    ('gb', HistGradientBoostingClassifier(loss='log_loss', random_state=42)),
])

params = {
    'gb__learning_rate': np.logspace(-3, 0, 10),
    'gb__min_samples_leaf': [1, 2, 4, 6],
    'gb__max_depth': [3, 4, 5, 6, 8, 10, 20],
    'gb__max_features': np.arange(0.1, 1.1, 0.1),
    'gb__max_leaf_nodes': [None, 5, 10, 20],
    'gb__validation_fraction': [0.1, 0.15, 0.2],
}

gb_us_rs = RandomizedSearchCV(
    gb_us_pipeline,
    n_iter=200,
    param_distributions=params,
    scoring=scoring,
    cv=kf,
    refit='Balanced Accuracy',
    return_train_score=True,
    n_jobs=-1
)

gb_us_rs.fit(X_train, y_train)

print(gb_us_rs.best_params_)
print()
print_metrics_rs(gb_us_rs)

#### Round 2

Works better without undersampling and just class weight.

In [ ]:
rs_cv_results(gb_us_rs)

In [ ]:
gb_us_pipeline = ImbPipeline([
    ('rus', RandomUnderSampler()),
    ('contact_intensity', ContactIntensityFeature()),
    ('gb', HistGradientBoostingClassifier(
        loss='log_loss',
        random_state=42
    )),
])

params = {
    'gb__learning_rate': np.logspace(-3, -0.3, 10),
    'gb__min_samples_leaf': [1, 2, 6],
    'gb__max_depth': [5, 10, 20],
    'gb__max_features': np.arange(0.1, 0.6, 0.05),
    'gb__max_leaf_nodes': [None, 5, 10, 20],
    'gb__validation_fraction': [0.1, 0.15],
    'gb__class_weight': [
        'balanced',
        {0: 1, 1: 50},
        {0: 1, 1: 75},
        {0: 1, 1: 100},
        {0: 1, 1: 125},
    ]
}

gb_rs = RandomizedSearchCV(
    gb_us_pipeline,
    n_iter=200,
    param_distributions=params,
    scoring=scoring,
    cv=kf,
    refit='Balanced Accuracy',
    return_train_score=True,
    n_jobs=-1
)

gb_rs.fit(X_train, y_train)

print(gb_rs.best_params_)
print()
print_metrics_rs(gb_rs)

In [ ]:
# The previous model worked better

joblib.dump(gb_us_rs.best_estimator_, '../models/bank/gb.joblib')

### XGB

In [ ]:
from xgboost import XGBClassifier

In [ ]:
xgb_pipeline = Pipeline([
    ('contact_intensity', ContactIntensityFeature()),
    ('xgb', XGBClassifier(n_jobs=-1)),
])

params = {
    'xgb__learning_rate': np.logspace(-2, 0, 10),
    'xgb__max_depth': [2, 3, 4, 5, 8, 10],
    'xgb__min_child_weight': [4, 6, 8, 10, 12],
    'xgb__colsample_bytree': np.arange(0.05, 0.6, 0.05),
    'xgb__max_leaves': [10, 12, 14, 15, 20],
    'xgb__subsample': np.arange(0.5, 1, 0.05),
    'xgb__scale_pos_weight': [1, 25, 50, 75, 100, 125]
}

xgb_rs = RandomizedSearchCV(
    xgb_pipeline,
    n_iter=300,
    param_distributions=params,
    scoring=scoring,
    cv=kf,
    refit='Balanced Accuracy',
    return_train_score=True,
    n_jobs=-1
)

xgb_rs.fit(X_train, y_train)

print(xgb_rs.best_params_)
print()
print_metrics_rs(xgb_rs)

In [ ]:
# Extract the feature names from the preprocessor
# feature_names = xgb_rs.best_estimator_.named_steps['preprocessor'].get_feature_names_out()

xgb_ft_imps = pd.DataFrame({
    'feature': X_train.columns.to_list() + ['contact_intensity'],
    'importance': xgb_rs.best_estimator_['xgb'].feature_importances_
}).sort_values('importance', ascending=False).round(3)

xgb_ft_imps.head(20)

In [ ]:
joblib.dump(xgb_rs.best_estimator_, '../models/bank/xgb.joblib')

### Balanced RF

In [ ]:
from imblearn.ensemble import BalancedRandomForestClassifier

In [ ]:
brf_pipeline = Pipeline([
    ('contact_intensity', ContactIntensityFeature()),
    ('preprocessor', preprocessor),
    ('brf', BalancedRandomForestClassifier(
        replacement=True,
        random_state=42,
        n_jobs=-1
    )),
])

params = [{
    'brf__sampling_strategy': [0.1, 0.25, 0.5, 0.75, 1, 'not minority', 'all'],
    'brf__n_estimators': [100, 200, 300],
    'brf__min_samples_split': [2, 5, 10, 20],
    'brf__min_samples_leaf': [1, 2, 4, 6],
    'brf__max_depth': [5, 10, 30, 50, None],
    'brf__max_samples': [None, 0.5, 0.7, 0.9],
    'brf__max_leaf_nodes': [None, 5, 10, 20],
}]

brf_rs = RandomizedSearchCV(
    brf_pipeline,
    n_iter=200,
    param_distributions=params,
    scoring=scoring,
    cv=kf,
    refit='Balanced Accuracy',
    return_train_score=True
)

brf_rs.fit(X_train, y_train)

print(brf_rs.best_params_)
print()
print_metrics_rs(brf_rs)

In [ ]:
# Extract the feature names from the preprocessor
feature_names = rf_rs.best_estimator_.named_steps['preprocessor'].get_feature_names_out()

brf_ft_imps = pd.DataFrame({
    'feature': feature_names,
    'importance': brf_rs.best_estimator_['brf'].feature_importances_
}).sort_values('importance', ascending=False).round(3)

brf_ft_imps.head(20)

In [ ]:
joblib.dump(brf_rs.best_estimator_, '../models/bank/brf.joblib')

#### Full train study

In [ ]:
brf = joblib.load('../models/bank/brf.joblib')

In [ ]:
# get the probabilities for the positive class in the validation sets
y_val_probs = cross_val_predict(brf, X_train, y_train, cv=kf, method='predict_proba')
y_val_probs_1 = y_val_probs[:, 1]

In [ ]:
idx0 = np.where(y_train == 0)[0]
idx1 = np.where(y_train == 1)[0]

plt.boxplot([y_val_probs_1[idx0], y_val_probs_1[idx1]], tick_labels=[0, 1])
plt.xlabel('Real class')
plt.ylabel('Predicted Probability of class 1')
plt.show()

In [ ]:
# Vary the decision threshold and show Precision and Recall

print('Threshold | B. Acc.    | Recall    | Precision')
print('-----------------------------------------------------')
for th in list(np.arange(0, 1, 0.05)) + [0.9999]:
    bacc = balanced_accuracy_score(y_train, y_val_probs_1 > th)
    rec = recall_score(y_train, y_val_probs_1 > th)
    prec = precision_score(y_train, y_val_probs_1 > th, zero_division=0)
    print(f'{th:.2f} \t  | {prec:.3f}      | {rec:.2f}      | {bacc:.2f}')

In [ ]:
# Vary the decision on both class 1 and 0, removing data for which the model is less sure

print(f'{"Threshold":>9} | {"% Data":>7} | {"B. Acc.":>7} | {"Recall":>7} | {"Precision":>9}')
print('-' * 55)

for th in np.arange(0, 0.99, 0.05):
    y_val_ser = pd.Series(y_val_probs_1, index=y_train.index)
    likely_df = y_val_ser[(y_val_ser > th) | (y_val_ser < 1 - th)]
    likely_pred = (likely_df >= 0.5).astype(int)
    likely_y = y_train.loc[likely_df.index]

    perc_data = len(likely_df) / len(y_train)
    bacc = balanced_accuracy_score(likely_y, likely_pred)
    rec = recall_score(likely_y, likely_pred)
    prec = precision_score(likely_y, likely_pred, zero_division=0)

    print(f'{th:9.2f} | {perc_data:7.0%} | {bacc:7.3f} | {rec:7.3f} | {prec:9.3f}')

### TabPFN

In [ ]:
tabpfn_pipeline = Pipeline([
    ('contact_intensity', ContactIntensityFeature()),
    ('tabpfn', TabPFNClassifier(
        model_path='models/tabpfn_v3/tabpfn-v3-classifier-v3_20260417_binary.ckpt',
        device='cpu',
        ignore_pretraining_limits=True,   # to allow running on more than 1000 rows with cpu
        n_jobs=-1,
    )),
])

# Perform cross-validation (no need of hyperparameter tuning)
tabpfn_cv = cross_validate(
    tabpfn_pipeline,
    X_train,
    y_train,
    cv=kf,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

print_metrics(tabpfn_cv)

In [ ]:
# get the probabilities for the positive class in the validation sets
y_val_probs = cross_val_predict(tabpfn_pipeline, X_train, y_train, cv=kf, method='predict_proba')
y_val_probs_1 = y_val_probs[:, 1]

In [ ]:
idx0 = np.where(y_train == 0)[0]
idx1 = np.where(y_train == 1)[0]

plt.boxplot([y_val_probs_1[idx0], y_val_probs_1[idx1]], tick_labels=[0, 1])
plt.xlabel('Real class')
plt.ylabel('Predicted Probability of class 1')
plt.show()

By varying the decision threshold, it achieves a good balanced accuracy!

In [ ]:
# Vary the decision threshold and show Precision and Recall

print('Threshold | B. Acc.    | Recall    | Precision')
print('-----------------------------------------------------')
for th in list(np.arange(0, 1, 0.05)) + [0.9999]:
    bacc = balanced_accuracy_score(y_train, y_val_probs_1 > th)
    rec = recall_score(y_train, y_val_probs_1 > th)
    prec = precision_score(y_train, y_val_probs_1 > th, zero_division=0)
    print(f'{th:.2f} \t  | {prec:.3f}      | {rec:.2f}      | {bacc:.2f}')

In [ ]:
joblib.dump(tabpfn_pipeline, '../models/bank/tabpfn.joblib')

## Test

In [ ]:
models = {
    'LR': joblib.load('../models/bank/lr.joblib'),
    'RF': joblib.load('../models/bank/rf_ru.joblib'),
    'GB': joblib.load('../models/bank/gb.joblib'),
    'XGB': joblib.load('../models/bank/xgb.joblib'),
    'BRF': joblib.load('../models/bank/brf.joblib'),
    'TabPFN': joblib.load('../models/bank/tabpfn.joblib'),
}

Based on validation balanced accuracy, search for the best decision threshold for each model.

In [ ]:
thresholds = np.arange(0, 1.01, 0.01)

best_thresholds = {}
test_bal_accs = {}

for name, model in models.items():
    print(name)

    # Validation preds via CV
    val_probs1 = cross_val_predict(
        model,
        X_train,
        y_train,
        cv=kf,
        method="predict_proba"
    )[:, 1]

    # Scan thresholds
    best_bacc = -1
    best_th = None

    for th in thresholds:
        preds = (val_probs1 >= th).astype(int)
        bacc = balanced_accuracy_score(y_train, preds)
        if bacc > best_bacc:
            best_bacc = bacc
            best_th = th

    best_thresholds[name] = best_th

    # Retrain model fully
    model.fit(X_train, y_train)

    # Evaluate on test using chosen threshold
    test_probs = model.predict_proba(X_test)[:, 1]
    test_preds = (test_probs >= best_th).astype(int)

    test_bacc = balanced_accuracy_score(y_test, test_preds)
    test_bal_accs[name] = test_bacc

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(test_bal_accs.keys(), test_bal_accs.values())
plt.ylabel("Balanced Accuracy (Test)")
plt.title("Model Comparison with Best Threshold per Model")
plt.ylim(0, 1)
plt.show()

In [ ]:
test_bal_accs

In [ ]:
best_th = best_thresholds['TabPFN']

test_probs = models['TabPFN'].predict_proba(X_test)[:, 1]
test_preds = (test_probs >= best_th).astype(int)

cm = confusion_matrix(y_test, test_preds, labels=[0, 1])
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[0, 1]
)
disp.plot()
plt.show()

## Submission

In [ ]:
thresholds = np.arange(0, 1.01, 0.01)

# Validation preds via CV
probs1 = cross_val_predict(
    models['TabPFN'],
    X,
    y,
    cv=kf,
    method="predict_proba"
)[:, 1]
    
# Scan thresholds
best_bacc = -1
best_th = None

for th in thresholds:
    preds = (probs1 >= th).astype(int)
    bacc = balanced_accuracy_score(y, preds)
    if bacc > best_bacc:
        best_bacc = bacc
        best_th = th


print('Best B. Acc.:', round(best_bacc, 3))
print('Best threshold:', best_th)

In [ ]:
# The function trains with the whole dataset

create_submission(
    models['TabPFN'],
    X,
    y,
    leaderboard_df,
    fts2drop,
    decision_th=best_th,
    fpath='../data/bank_term_deposit/tabpfn_bank_submission.csv'
)